In [ ]:
import os, sys, json
from concurrent.futures import ThreadPoolExecutor
sys.path.append('..')
from dotenv import load_dotenv
import anthropic
from utils.helpers import CLAUDE_SONNET, CLAUDE_HAIKU, CLAUDE_OPUS, DEFAULT_MODEL

load_dotenv()
client = anthropic.Anthropic()
MODEL = DEFAULT_MODEL
MODEL_HAIKU = CLAUDE_HAIKU  # Para tareas simples

def llm(prompt: str, system: str = "", model: str = MODEL, max_tokens: int = 1024) -> str:
    kwargs = {"model": model, "max_tokens": max_tokens,
              "messages": [{"role": "user", "content": prompt}]}
    if system:
        kwargs["system"] = system
    return client.messages.create(**kwargs).content[0].text


## 1. Parallelization Workflow

In [ ]:
if "client" not in dir():
    import sys, time; sys.path.append('..')
    from dotenv import load_dotenv
    from concurrent.futures import ThreadPoolExecutor
    import anthropic
    from utils.helpers import CLAUDE_HAIKU, DEFAULT_MODEL
    load_dotenv()
    client = anthropic.Anthropic()
    MODEL = DEFAULT_MODEL
    MODEL_HAIKU = CLAUDE_HAIKU
    def llm(prompt: str, system: str = "", model: str = MODEL, max_tokens: int = 1024) -> str:
        kwargs = {"model": model, "max_tokens": max_tokens,
                  "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        return client.messages.create(**kwargs).content[0].text
import time

# Caso de uso: analizar un artículo desde múltiples perspectivas en paralelo

articulo = """
La empresa TechGlobal anunció hoy el lanzamiento de su nueva plataforma de IA 
que promete revolucionar el sector educativo. La plataforma utiliza modelos de 
lenguaje avanzados para personalizar el aprendizaje de cada estudiante. 
La inversión inicial es de $50 millones y esperan llegar a 1 millón de usuarios 
en el primer año. Los críticos señalan preocupaciones sobre privacidad de datos 
de menores y el posible reemplazo de docentes.
"""

# Tareas paralelas
analisis_tasks = [
    ("resumen",     "Resume el artículo en 2 oraciones."),
    ("sentimiento", "¿Cuál es el tono del artículo: positivo, negativo o neutral? Justifica."),
    ("entidades",   "Lista las entidades mencionadas: empresa, cantidades, personas."),
    ("riesgos",     "¿Qué riesgos o problemas se mencionan en el artículo?"),
]

def analizar(task):
    nombre, pregunta = task
    result = llm(f"""
Artículo: {articulo}

Tarea: {pregunta}
""", model=MODEL_HAIKU, max_tokens=200)
    return nombre, result

# Secuencial
t0 = time.time()
for task in analisis_tasks:
    analizar(task)
t_sec = time.time() - t0

# Paralelo
t0 = time.time()
with ThreadPoolExecutor(max_workers=4) as executor:
    resultados = dict(executor.map(analizar, analisis_tasks))
t_par = time.time() - t0

print(f"Secuencial: {t_sec:.2f}s | Paralelo: {t_par:.2f}s")
print(f"Speedup: {t_sec/t_par:.1f}x mas rapido")
print()
for nombre, resultado in resultados.items():
    print(f"[{nombre.upper()}]")
    print(resultado.strip()[:200])
    print()


## 2. Chaining Workflow

In [ ]:
# Cadena: Ideas → Seleccionar → Desarrollar → Revisar → Publicar
# Cada etapa usa el output de la anterior

tema = "aplicaciones de IA en la educación rural"
print(f"Tema: {tema}\n")

# Etapa 1: Generar ideas
print("[1/4] Generando ideas...")
ideas = llm(f"""
Genera 5 ideas originales para un artículo de blog sobre: {tema}
Formato: solo los títulos, numerados.
""", model=MODEL_HAIKU, max_tokens=300)
print(ideas)

# Etapa 2: Seleccionar la mejor
print("\n[2/4] Seleccionando la mejor idea...")
mejor_idea = llm(f"""
De estas ideas para un artículo:
{ideas}

Selecciona la más original e impactante. Responde SOLO con el título elegido.
""", model=MODEL_HAIKU, max_tokens=100)
print(f"✅ Elegida: {mejor_idea.strip()}")

# Etapa 3: Desarrollar el contenido
print("\n[3/4] Desarrollando el artículo...")
borrador = llm(f"""
Escribe un artículo de blog de 200 palabras sobre:
{mejor_idea}

Incluye: introducción, 2-3 puntos clave y conclusión.
""", max_tokens=400)
print(borrador[:300] + "...")

# Etapa 4: Revisión editorial
print("\n[4/4] Revisión editorial...")
revision = llm(f"""
Revisa este borrador y mejora:
1. El gancho inicial (más impactante)
2. La claridad del mensaje
3. El llamado a la acción final

Borrador:
{borrador}

Devuelve solo el artículo mejorado.
""", max_tokens=500)

print("\n=== ARTÍCULO FINAL ===")
print(revision)

## 3. Routing Workflow

In [ ]:
# Router: clasifica y envía al agente especializado correcto

AGENTES = {
    "TECNICO": "Eres un experto técnico en software y hardware. Responde de forma técnica y precisa.",
    "FACTURACION": "Eres especialista en facturación y pagos. Ayuda con cobros, facturas y cuentas.",
    "VENTAS": "Eres vendedor entusiasta. Resalta beneficios y ofrece opciones de compra.",
    "GENERAL": "Eres asistente general amigable y útil."
}

def router(consulta: str) -> str:
    """Clasifica la consulta y retorna el agente adecuado."""
    prompt = f"""
Clasifica esta consulta de soporte en una categoría:
- TECNICO: problemas de software, hardware, errores
- FACTURACION: pagos, facturas, cobros, reembolsos
- VENTAS: compras, precios, planes, upgrades
- GENERAL: otras consultas

Consulta: {consulta}

Responde SOLO con la categoría en MAYÚSCULAS.
"""
    return llm(prompt, model=MODEL_HAIKU, max_tokens=20).strip().upper()

def atender_consulta(consulta: str) -> str:
    """Router completo: clasifica y responde con el agente correcto."""
    categoria = router(consulta)
    # Normalizar categoría
    if categoria not in AGENTES:
        categoria = "GENERAL"
    
    system = AGENTES[categoria]
    respuesta = llm(consulta, system=system, model=MODEL_HAIKU, max_tokens=200)
    return categoria, respuesta

# Prueba con diferentes tipos de consultas
consultas = [
    "Mi aplicación se crashea cuando subo archivos grandes",
    "Me cobraron dos veces este mes, ¿qué hago?",
    "¿Cuál plan me conviene si tengo un equipo de 10 personas?",
    "¿Cómo cambio mi contraseña?",
]

for consulta in consultas:
    categoria, respuesta = atender_consulta(consulta)
    print(f"\n👤 Cliente: {consulta}")
    print(f"📌 Ruta → Agente {categoria}")
    print(f"🤖 Respuesta: {respuesta[:200]}")
    print("-" * 60)

## 4. Agente Autónomo con Herramientas

In [ ]:
import math

# Herramientas del agente
AGENT_TOOLS = [
    {
        "name": "calcular",
        "description": "Evalúa expresiones matemáticas. Usa funciones de Python: pow(), sqrt(), abs(), etc.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expresion": {"type": "string", "description": "Expresión matemática Python"}
            },
            "required": ["expresion"]
        }
    },
    {
        "name": "buscar_info",
        "description": "Busca información en la base de conocimiento.",
        "input_schema": {
            "type": "object",
            "properties": {
                "tema": {"type": "string", "description": "Tema a buscar"}
            },
            "required": ["tema"]
        }
    },
    {
        "name": "guardar_resultado",
        "description": "Guarda un resultado o nota en la memoria del agente.",
        "input_schema": {
            "type": "object",
            "properties": {
                "clave": {"type": "string"},
                "valor": {"type": "string"}
            },
            "required": ["clave", "valor"]
        }
    }
]

# Memoria del agente
agent_memory = {}

KB = {
    "python": "Python 3.12 es el lenguaje más popular para IA/ML. Lanzado en 2023.",
    "claude": "Claude es un asistente de IA de Anthropic. Versión actual: Claude Sonnet 4.5.",
    "matemáticas": "Las matemáticas son la base de la computación y la IA."
}

def ejecutar_herramienta_agente(nombre: str, inputs: dict) -> str:
    if nombre == "calcular":
        try:
            safe_ns = {k: v for k, v in math.__dict__.items() if not k.startswith('_')}
            safe_ns['pow'] = pow
            result = eval(inputs['expresion'], {"__builtins__": {}}, safe_ns)
            return json.dumps({"resultado": result})
        except Exception as e:
            return json.dumps({"error": str(e)})
    
    elif nombre == "buscar_info":
        tema = inputs['tema'].lower()
        for key, val in KB.items():
            if key in tema:
                return json.dumps({"encontrado": val})
        return json.dumps({"encontrado": "No se encontró información sobre ese tema."})
    
    elif nombre == "guardar_resultado":
        agent_memory[inputs['clave']] = inputs['valor']
        return json.dumps({"guardado": True})

def run_agent(objetivo: str, max_pasos: int = 10) -> str:
    """Ejecuta el agente autónomo hasta completar el objetivo."""
    print(f"🎯 Objetivo: {objetivo}\n")
    messages = [{"role": "user", "content": objetivo}]
    paso = 0
    
    while paso < max_pasos:
        paso += 1
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            system="""Eres un agente autónomo. Usa las herramientas disponibles 
para completar el objetivo del usuario. Sé eficiente y preciso.""",
            tools=AGENT_TOOLS,
            messages=messages
        )
        
        print(f"[Paso {paso}] stop_reason: {response.stop_reason}")
        
        if response.stop_reason == "end_turn":
            final = " ".join(b.text for b in response.content if hasattr(b, 'text'))
            print(f"\n✅ Tarea completada:\n{final}")
            return final
        
        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 {block.name}({json.dumps(block.input)[:60]})")
                    result = ejecutar_herramienta_agente(block.name, block.input)
                    print(f"  ← {result[:80]}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })
            messages.append({"role": "user", "content": tool_results})
    
    return "Límite de pasos alcanzado"

# Ejecutar el agente
run_agent("""
Necesito que:
1. Calcules cuántos segundos hay en una semana
2. Busques info sobre Claude
3. Guardes ambos resultados en memoria
4. Me presentes un resumen final
""")